# Exercices : Se connecter à une base de données

Ce notebook accompagne le cours **Se connecter à une base de données**. Les exercices ne réexpliquent pas les notions : si tu bloques, retourne voir le cours.

Le fil rouge : la base des **commandes d'un entrepôt**. Complète les cellules marquées `# TODO` et exécute-les **dans l'ordre** : chaque exercice utilise la connexion ouverte au début.

---

## Préparation

Exécute cette cellule : elle supprime la base si elle existe déjà, pour que tu puisses relancer le notebook depuis le début autant de fois que tu veux.

In [ ]:
import os
import sqlite3

if os.path.exists("entrepot.db"):
    os.remove("entrepot.db")

commandes = [
    ("Durand",  "Lyon",      "Palette", 10, 12.5),
    ("Martin",  "Lille",     "Carton",  200, 0.8),
    ("Durand",  "Lille",     "Film",    15, 6.0),
    ("Bernard", "Lyon",      "Carton",  150, 0.8),
    ("Martin",  "Lyon",      "Palette", 4, 12.5),
    ("Petit",   "Marseille", "Film",    30, 6.0),
    ("Bernard", "Marseille", "Palette", 8, 12.5),
]

---

## Exercice 1 : Créer la base et la table

Ouvre une connexion vers le fichier `entrepot.db`, puis crée une table `commandes` avec les colonnes :

- `id` : identifiant unique rempli automatiquement ;
- `client`, `entrepot`, `produit` : du texte, obligatoire ;
- `quantite` : un entier ;
- `prix_unitaire` : un nombre décimal.

In [ ]:
# TODO : ouvre la connexion et crée la table
connexion = ...

print(connexion.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall())
# attendu : [('commandes',)]

---

## Exercice 2 : Insérer les commandes

Insère **en une seule instruction** toutes les commandes de la liste `commandes`, puis enregistre les modifications. Vérifie ensuite le nombre de lignes avec une requête `COUNT(*)`, en récupérant **un seul** résultat.

In [ ]:
# TODO : insère les commandes et enregistre


# TODO : compte les lignes de la table
nb = ...
print(nb)
# attendu : (7,)

---

## Exercice 3 : Lire et parcourir un résultat

Affiche les commandes de l'entrepôt de **Lyon**, une par ligne, au format ci-dessous. Parcours directement le résultat avec une boucle `for`, sans `fetchall()`.

In [ ]:
# TODO
...

# attendu :
# Durand   Palette  x10
# Bernard  Carton   x150
# Martin   Palette  x4

---

## Exercice 4 : Faire calculer la base

Sans aucune boucle Python, obtiens le **montant total** (quantité × prix unitaire) de chaque entrepôt, du plus gros au plus petit.

In [ ]:
# TODO
requete = ...

print(connexion.execute(requete).fetchall())
# attendu : [('Lyon', 295.0), ('Marseille', 280.0), ('Lille', 250.0)]

---

## Exercice 5 : `commit()` ou `rollback()` ?

Un collègue a lancé un nettoyage... et s'est trompé de condition. Exécute la cellule : elle supprime des lignes **sans les enregistrer**.

Complète-la pour **annuler** la suppression, puis vérifie qu'il y a toujours 7 commandes.

In [ ]:
connexion.execute("DELETE FROM commandes WHERE quantite > 5")
print(connexion.execute("SELECT COUNT(*) FROM commandes").fetchone())   # la suppression est visible...

# TODO : annule la suppression
...

print(connexion.execute("SELECT COUNT(*) FROM commandes").fetchone())
# attendu : (1,) puis (7,)

---

## Exercice 6 : Repérer et corriger une injection

Cette fonction renvoie les commandes d'un client. Exécute la cellule : avec une saisie piégée, elle renvoie **toutes** les commandes.

Corrige **la fonction** (pas l'appel) pour qu'elle soit protégée.

In [ ]:
def commandes_client(connexion, client):
    requete = f"SELECT client, produit FROM commandes WHERE client = '{client}'"
    return connexion.execute(requete).fetchall()

print(commandes_client(connexion, "Petit"))
print(len(commandes_client(connexion, "Petit' OR '1'='1")))
# attendu après correction :
# [('Petit', 'Film')]
# 0

---

## Exercice 7 : Plusieurs paramètres

Écris une fonction `grosses_commandes(connexion, entrepot, quantite_min)` qui renvoie le client et la quantité des commandes d'un entrepôt dont la quantité est **au moins** égale à `quantite_min`. Les deux valeurs passent par des paramètres.

In [ ]:
# TODO
...

print(grosses_commandes(connexion, "Lyon", 10))
print(grosses_commandes(connexion, "Lille", 500))
# attendu :
# [('Durand', 10), ('Bernard', 150)]
# []

---

## Exercice 8 : Passer à SQLAlchemy

Refais la requête de l'exercice précédent avec **SQLAlchemy** : un engine vers `entrepot.db`, un paramètre **nommé** pour l'entrepôt et un autre pour la quantité minimale.

Cherche les commandes de **Marseille** d'au moins **5** unités.

In [ ]:
from sqlalchemy import create_engine, text

# TODO
engine = ...

# attendu : [('Petit', 30), ('Bernard', 8)]

---

## Exercice 9 : Charger dans pandas

Charge dans un DataFrame `df_palettes` toutes les commandes de **palettes**, avec une requête paramétrée. Affiche ensuite la quantité totale commandée, calculée avec pandas.

In [ ]:
import pandas as pd

# TODO
df_palettes = ...

print(df_palettes[["client", "entrepot", "quantite"]])
print(df_palettes["quantite"].sum())
# attendu :
#     client   entrepot  quantite
# 0   Durand       Lyon        10
# 1   Martin       Lyon         4
# 2  Bernard  Marseille         8
# 22

---

## Exercice 10 : Mini-projet : un rapport enregistré dans la base

Le responsable de l'entrepôt veut une table `rapport_clients` dans la base, avec pour chaque client son **nombre de commandes** et son **montant total**, triée par montant décroissant.

1. Fais calculer ces chiffres **par la base** et récupère-les dans un DataFrame `rapport` (pense à la règle : filtrer et agréger dans la base).
2. Enregistre `rapport` dans la table `rapport_clients`, en remplaçant la table si elle existe et sans enregistrer l'index.
3. Relis la table pour vérifier, puis **ferme la connexion**.

In [ ]:
# TODO 1
rapport = ...

# TODO 2


# TODO 3
print(pd.read_sql("SELECT * FROM rapport_clients", connexion))
# attendu :
#     client  nb_commandes  montant
# 0  Bernard             2    220.0
# 1   Durand             2    215.0
# 2   Martin             2    210.0
# 3    Petit             1    180.0

---

## Corrigés

<details>
<summary>Afficher les corrigés</summary>

**Exercice 1**
```python
connexion = sqlite3.connect("entrepot.db")

connexion.execute("""
    CREATE TABLE IF NOT EXISTS commandes (
        id INTEGER PRIMARY KEY,
        client TEXT NOT NULL,
        entrepot TEXT NOT NULL,
        produit TEXT NOT NULL,
        quantite INTEGER,
        prix_unitaire REAL
    )
""")

print(connexion.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall())
```
`sqlite_master` est une table interne de SQLite qui liste le contenu de la base.

**Exercice 2**
```python
connexion.executemany(
    "INSERT INTO commandes (client, entrepot, produit, quantite, prix_unitaire) VALUES (?, ?, ?, ?, ?)",
    commandes,
)
connexion.commit()

nb = connexion.execute("SELECT COUNT(*) FROM commandes").fetchone()
print(nb)
```
`fetchone()` renvoie un tuple, même quand la requête ne renvoie qu'une valeur : `nb[0]` donne `7`.

**Exercice 3**
```python
for client, produit, quantite in connexion.execute(
    "SELECT client, produit, quantite FROM commandes WHERE entrepot = 'Lyon'"
):
    print(f"{client:<8} {produit:<8} x{quantite}")
```

**Exercice 4**
```python
requete = """
    SELECT entrepot, SUM(quantite * prix_unitaire) AS montant
    FROM commandes
    GROUP BY entrepot
    ORDER BY montant DESC
"""

print(connexion.execute(requete).fetchall())
```

**Exercice 5**
```python
connexion.execute("DELETE FROM commandes WHERE quantite > 5")
print(connexion.execute("SELECT COUNT(*) FROM commandes").fetchone())

connexion.rollback()

print(connexion.execute("SELECT COUNT(*) FROM commandes").fetchone())
```
Tant que rien n'est enregistré, la connexion voit ses propres modifications en cours. `rollback()` annule tout depuis le dernier `commit()`.

**Exercice 6**
```python
def commandes_client(connexion, client):
    requete = "SELECT client, produit FROM commandes WHERE client = ?"
    return connexion.execute(requete, (client,)).fetchall()

print(commandes_client(connexion, "Petit"))
print(len(commandes_client(connexion, "Petit' OR '1'='1")))
```
Avec le paramètre, la base cherche un client qui s'appelle littéralement `Petit' OR '1'='1` : il n'y en a aucun. N'oublie pas la virgule de `(client,)`.

**Exercice 7**
```python
def grosses_commandes(connexion, entrepot, quantite_min):
    requete = "SELECT client, quantite FROM commandes WHERE entrepot = ? AND quantite >= ?"
    return connexion.execute(requete, (entrepot, quantite_min)).fetchall()

print(grosses_commandes(connexion, "Lyon", 10))
print(grosses_commandes(connexion, "Lille", 500))
```
Les `?` sont remplis dans l'ordre du tuple : l'ordre compte.

**Exercice 8**
```python
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///entrepot.db")

with engine.connect() as conn:
    resultat = conn.execute(
        text("SELECT client, quantite FROM commandes WHERE entrepot = :entrepot AND quantite >= :qmin"),
        {"entrepot": "Marseille", "qmin": 5},
    )
    print(resultat.fetchall())
```
Le reste du code serait identique avec PostgreSQL : seule l'URL de `create_engine()` changerait.

**Exercice 9**
```python
import pandas as pd

df_palettes = pd.read_sql(
    "SELECT * FROM commandes WHERE produit = ?",
    connexion,
    params=("Palette",),
)

print(df_palettes[["client", "entrepot", "quantite"]])
print(df_palettes["quantite"].sum())
```

**Exercice 10**
```python
rapport = pd.read_sql(
    """
    SELECT client,
           COUNT(*) AS nb_commandes,
           SUM(quantite * prix_unitaire) AS montant
    FROM commandes
    GROUP BY client
    ORDER BY montant DESC
    """,
    connexion,
)

rapport.to_sql("rapport_clients", connexion, if_exists="replace", index=False)

print(pd.read_sql("SELECT * FROM rapport_clients", connexion))
connexion.close()
```
Si tu relances la cellule, `if_exists="replace"` évite l'erreur « la table existe déjà ».

</details>